In [4]:
from bs4 import BeautifulSoup
import re
import json

In [5]:
TRANSLIT = {
    # Consonnes
    '\u05D0': "'",
    '\u05D1': 'b',
    '\u05D2': 'g',
    '\u05D3': 'd',
    '\u05D4': 'h',
    '\u05D5': 'w',
    '\u05D6': 'z',
    '\u05D7': 'kh',
    '\u05D8': 't',
    '\u05D9': 'y',
    '\u05DB': 'k',
    '\u05DA': 'k',
    '\u05DC': 'l',
    '\u05DE': 'm',
    '\u05DD': 'm',
    '\u05E0': 'n',
    '\u05DF': 'n',
    '\u05E1': 's',
    '\u05E2': "'",
    '\u05E4': 'p',
    '\u05E3': 'p',
    '\u05E6': 'ts',
    '\u05E5': 'ts',
    '\u05E7': 'q',
    '\u05E8': 'r',
    '\u05E9': 'sh',
    '\u05EA': 't',
    # Voyelles (niqqud)
    '\u05B0': 'e', '\u05B1': 'e', '\u05B2': 'a', '\u05B3': 'o',
    '\u05B4': 'i', '\u05B5': 'e', '\u05B6': 'e', '\u05B7': 'a',
    '\u05B8': 'a', '\u05B9': 'o', '\u05BA': 'o', '\u05BB': 'u',
    '\u05BC': '',  # dagesh (ignoré)
    '\u05BE': '-', # maqqef (tiret)
    '\u05C1': '',  # shin dot
    '\u05C2': '',  # sin dot
}

In [ ]:
def strip_vowels(text): #Supprime les voyelles de 'text'
    return re.sub(r'[\u05B0-\u05C7\u05F0-\u05F4\uFB1D-\uFB4E]', '', text)

def extract_headwords(p): #Extrait les entrées hébreux du dictionnaire
    words = []
    for tag in p.children:
        if tag.name == 'strong':
            break
        if tag.name == 'span' and tag.get('class') == ['hebrew']:
            words.append(tag.text)
    return words

def transliterate(text, drop_gutturals=False):
    text = text.replace('ש\u05C1', 'sh').replace('ש\u05C2', 's')
    result = ''
    for char in text:
        if drop_gutturals and char in ('\u05D0', '\u05E2'):
            result += ''
        else:
            result += TRANSLIT.get(char, char)
    return result

def parse_entry(p, sort_key): #extrait toutes les entrées et les stocke
    page = int(p.find('span').text.strip().rstrip(':'))
    strongs_span = p.find('span', class_='strongs-number')
    strongs = strongs_span.text if strongs_span else None
    headwords = extract_headwords(p)
    return {
        "page": page,
        "strongs": strongs,
        "m": headwords,
        "h": [strip_vowels(w) for w in headwords],
        "l": [t for w in headwords for t in (transliterate(w), transliterate(w, drop_gutturals=True))],
        "d": str(p),
        "sort_key": sort_key
    }

In [7]:
# ouvre les fichier html et le clean des <p> qui ne correspondet pas à des entrées de mot
with open('BDB_dict.html', encoding='utf-8') as f:
    html = f.read()

html = re.sub(r'</p>\s*<p>(?!<span>\d+:)', ' ', html) #Clean le html lorsqu'il y a plusieurs <p> dans une même entrée.

soup = BeautifulSoup(html, features='html.parser')

In [9]:
#Stocke tous les p dans entries
ps = soup.find_all('p')
entries = [p for p in ps if p.find('span') and re.match(r'^\d+:$', p.find('span').text.strip())]
print(f"Entrées trouvées : {len(entries)}")

Entrées trouvées : 11772


In [10]:
bdb = []
for i, p in enumerate(entries):
    bdb.append(parse_entry(p, i+1))

print(f"Entrées créées : {len(bdb)}")
print(bdb[10])

Entrées créées : 11772
{'page': 2, 'strongs': '12, 13', 'm': ['אַבྰדָָן', 'אָבྰדַן'], 'h': ['אבྰדן', 'אבྰדן'], 'l': ["'abྰdaan", 'abྰdaan', "'abྰdan", 'abྰdan'], 'd': '<p><span>2:</span> <span class="strongs-number">12, 13</span><span class="hebrew">אַבྰדָָן</span><span class="hebrew">אָבྰדַן</span><strong>n.[m.]</strong> destruction (Syr. 0’abdono0) <span 9:5="" class="ref" data-ref="Es">Est 9:5</span><span class="hebrew">לַטַּתחֶ־רֶבוְ הֶרֶגוְ א</span><span 8:6="" class="ref" data-ref="Es">8:6</span></p>', 'sort_key': 11}


In [ ]:
# Enregistre le json
with open('BDB.json', 'w', encoding='utf-8') as f:
    json.dump(bdb, f, ensure_ascii=False, indent=2)

print("Fichier BDB.json créé")

Fichier BDB.json créé


In [ ]:
for entry in bdb[:20]:
    for m, l in zip(entry['m'], entry['l']):
        print(f"{m} → {l}")

א → '
אָב → 'ab
אבה → 'bh
אבב → 'bb
אֵב → 'eb
אָבִיב → 'abiyb
אֲבִגַיִל → 'abigayil
אֲבִיגַיִל → 'abiygayil
אבה → 'bh
אֲבַגְתָא → 'abageta'
אָבַד → 'abad
אֹבֵד → 'obed
אֲבֵדָה → 'abedah
אֲבַדֹּה → 'abadoh
אֲבַדּוֺ → 'abadwo
אבדון → 'bdwn
אַבྰדָָן → 'abྰdaan
אָבྰדַן → 'abྰdan
אֲבַדּוֺן → 'abadwon
אָבָה → 'abah
אֶבྰיוֺן → 'ebྰywon
אֲבִיּוֺנָה → 'abiywonah
אֵ֫בֶה → '֫ebeh
אבה → 'bh
אָב → 'ab
אֲבִי → 'abiy
אבה → 'bh
אָב → 'ab
אָב → 'ab
אָב → 'ab
אֲבִיעַ־לྰבוֺן → 'abiy'a-lྰbwon
אֲבִיאֵל → 'abiy'el


In [14]:
"""
Télécharge les entrées BDB depuis Sefaria.
Essaie d'abord l'API /api/lexicon/, puis fallback sur /api/words/ lettre par lettre.
"""
import requests
import json
import time

BASE = "https://www.sefaria.org"
HEADERS = {"User-Agent": "Mozilla/5.0"}
OUT = "bdb_complete.json"

def get(url):
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

# ── Option A: endpoint /api/lexicon/ ─────────────────────────────────────────
def try_lexicon_api():
    print("Tentative via /api/lexicon/BDB Dictionary ...")
    entries = []
    page = 0
    while True:
        data = get(f"{BASE}/api/lexicon/BDB%20Dictionary?page={page}")
        batch = data.get("entries", [])
        if not batch:
            break
        entries.extend(batch)
        print(f"  page {page}: {len(batch)} entrées (total: {len(entries)})")
        if not data.get("has_next", False):
            break
        page += 1
        time.sleep(0.3)
    return entries

# ── Option B: /api/words/ lettre par lettre ───────────────────────────────────
HEBREW_LETTERS = list("אבגדהוזחטיכלמנסעפצקרשת")

def try_words_api():
    print("Tentative via /api/words/ lettre par lettre ...")
    entries = []
    for letter in HEBREW_LETTERS:
        from urllib.parse import quote
        data = get(f"{BASE}/api/words/{quote(letter)}?always_split=1")
        bdb = [e for e in data if e.get("parent_lexicon") == "BDB Dictionary"]
        entries.extend(bdb)
        print(f"  {letter}: {len(bdb)} entrées (total: {len(entries)})")
        time.sleep(0.3)
    return entries

# ── Main ──────────────────────────────────────────────────────────────────────
entries = []

try:
    entries = try_lexicon_api()
except Exception as e:
    print(f"  Échec: {e}")

if not entries:
    try:
        entries = try_words_api()
    except Exception as e:
        print(f"  Échec: {e}")

if entries:
    with open(OUT, "w", encoding="utf-8") as f:
        json.dump(entries, f, ensure_ascii=False, indent=2)
    print(f"\n✓ {len(entries)} entrées sauvegardées dans {OUT}")
else:
    print("\n✗ Aucune entrée récupérée.")

Tentative via /api/lexicon/BDB Dictionary ...
  Échec: 404 Client Error: Not Found for url: https://www.sefaria.org/api/lexicon/BDB%20Dictionary?page=0
Tentative via /api/words/ lettre par lettre ...
  א: 0 entrées (total: 0)
  ב: 0 entrées (total: 0)
  ג: 0 entrées (total: 0)
  ד: 0 entrées (total: 0)
  ה: 0 entrées (total: 0)
  ו: 0 entrées (total: 0)
  ז: 0 entrées (total: 0)
  ח: 0 entrées (total: 0)
  ט: 0 entrées (total: 0)
  י: 0 entrées (total: 0)
  כ: 0 entrées (total: 0)
  ל: 0 entrées (total: 0)
  מ: 1 entrées (total: 1)
  נ: 0 entrées (total: 1)
  ס: 0 entrées (total: 1)
  ע: 0 entrées (total: 1)
  פ: 0 entrées (total: 1)
  צ: 0 entrées (total: 1)
  ק: 0 entrées (total: 1)
  ר: 0 entrées (total: 1)
  ש: 0 entrées (total: 1)
  ת: 0 entrées (total: 1)

✓ 1 entrées sauvegardées dans bdb_complete.json


In [29]:
import requests
import re
 
def browse(prefix=""):
    BUCKET = "sefaria-export"
    url = f"https://storage.googleapis.com/{BUCKET}?prefix={prefix}{'/' if prefix else ''}&delimiter=/"
    xml = requests.get(url).text
    for p in sorted(re.findall(r'<Prefix>([^<]+)</Prefix>', xml)):
        print(p)
    for k in sorted(re.findall(r'<Key>([^<]+)</Key>', xml)):
        print(k)
 
print("### TOP LEVEL ###")
browse()                                        # top-level

print("### JSON ###")
browse("json")          # dictionnaires
#browse("json/Reference/Dictionary/BDB/English/")      # BDB

print("### JSON/Reference ###")
browse("json/Reference")

print("### JSON/Reference/Dictionary ###")
browse("json/Reference/Dictionary")  

### TOP LEVEL ###
cltk-flat/
cltk-full/
json/
links/
misc/
schemas/
txt/
last_export.txt
table_of_contents.json
### JSON ###
json/
json/Chasidut/
json/Halakhah/
json/Jewish Thought/
json/Kabbalah/
json/Liturgy/
json/Midrash/
json/Mishnah/
json/Musar/
json/Reference/
json/Responsa/
json/Second Temple/
json/Talmud/
json/Tanakh/
json/Tosefta/
### JSON/Reference ###
json/Reference/
json/Reference/Dictionary/
json/Reference/Encyclopedic Works/
json/Reference/Grammar/
### JSON/Reference/Dictionary ###
json/Reference/Dictionary/
json/Reference/Dictionary/Hafla'ah ShebaArakhin on Sefer HeArukh/
json/Reference/Dictionary/Otzar La'azei Rashi/
json/Reference/Dictionary/Sefer HeArukh/


In [43]:
# Ce script import l'ensemble du dictionnaire BDB depuis l'api de sefaria.

import requests, json, time
from urllib.parse import quote

entries = {}
errors = []

# Commencer par la première entrée
current_ref = "BDB, רָם²"


with open("bdb_complete.json", encoding="utf-8") as f:
    entries = json.load(f)

while current_ref:
    url = f"https://www.sefaria.org/api/texts/{quote(current_ref)}"
    try:
        data = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15).json()
        entries[current_ref] = data
        next_ref = data.get("next")
        print(f"[{len(entries)}] {current_ref} → {next_ref}")
        current_ref = next_ref
        time.sleep(0.2)
    except Exception as e:
        errors.append((current_ref, str(e)))
        print(f"ERREUR {current_ref}: {e}")
        break

with open("bdb_complete.json", "w", encoding="utf-8") as f:
    json.dump(entries, f, ensure_ascii=False, indent=2)

print(f"\n✓ {len(entries)} entrées, {len(errors)} erreurs")

[9359] BDB, רָם² → BDB, רֵם
ERREUR BDB, רֵם: HTTPSConnectionPool(host='www.sefaria.org', port=443): Read timed out. (read timeout=15)

✓ 9359 entrées, 1 erreurs
